In [1]:
# imports
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, r2_score
import math

from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")
print("done")

done


## Data loading and preprocessing
### Load and display data
Specifying target column for easier use down the line

Can change said column later if wanted

In [2]:
# load data
target_value = "Anomaly Scores"
data = pd.read_csv("cybersecurity_attacks.csv")
data.head()

,Timestamp,Source IP Address,Destination IP Address,Source Port,Destination Port,Protocol,Packet Length,Packet Type,Traffic Type,Payload Data,...,Action Taken,Severity Level,User Information,Device Information,Network Segment,Geo-location Data,Proxy Information,Firewall Logs,IDS/IPS Alerts,Log Source
0,2023-05-30 06:33:58,103.216.15.12,84.9.164.252,31225,17616,ICMP,503,Data,HTTP,Qui natus odio asperiores nam. Optio nobis ius...,...,Logged,Low,Reyansh Dugal,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment A,"Jamshedpur, Sikkim",150.9.97.135,Log Data,NaN,Server
1,2020-08-26 07:08:30,78.199.217.198,66.191.137.154,17245,48166,ICMP,1174,Data,HTTP,Aperiam quos modi officiis veritatis rem. Omni...,...,Blocked,Low,Sumer Rana,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment B,"Bilaspur, Nagaland",NaN,Log Data,NaN,Firewall
2,2022-11-13 08:23:25,63.79.210.48,198.219.82.17,16811,53600,UDP,306,Control,HTTP,Perferendis sapiente vitae soluta. Hic delectu...,...,Ignored,Low,Himmat Karpe,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT ...,Segment C,"Bokaro, Rajasthan",114.133.48.179,Log Data,Alert Data,Firewall
3,2023-07-02 10:38:46,163.42.196.10,101.228.192.255,20018,32534,UDP,385,Data,HTTP,Totam maxime beatae expedita explicabo porro l...,...,Blocked,Medium,Fateh Kibe,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_11_5; ...,Segment B,"Jaunpur, Rajasthan",NaN,NaN,Alert Data,Firewall
4,2023-07-16 13:11:07,71.166.185.76,189.243.174.238,6131,26646,TCP,1462,Data,DNS,Odit nesciunt dolorem nisi iste iusto. Animi v...,...,Blocked,Low,Dhanush Chad,Mozilla/5.0 (compatible; MSIE 5.0; Windows NT ...,Segment C,"Anantapur, Tripura",149.6.110.119,NaN,Alert Data,Firewall


### Missing values
The data does contain missing values

Using mean imputation to fill values, could use something like knn or drop rows/columns instead if wanted

Dropping target column here as well

In [3]:
data.columns[data.isnull().sum() != 0]

Index(['Malware Indicators', 'Alerts/Warnings', 'Proxy Information',
       'Firewall Logs', 'IDS/IPS Alerts'],
      dtype='object')

In [4]:
num_null_counts = data.isnull().sum()
null_cols = data.columns[num_null_counts != 0]
if len(null_cols) > 0:
    print("Data contains NaN values")
    display(num_null_counts[num_null_counts != 0])

else:
    print("Data does not contain NaN values")


Data contains NaN values


Malware Indicators    20000
Alerts/Warnings       20067
Proxy Information     19851
Firewall Logs         19961
IDS/IPS Alerts        20050
dtype: int64

In [5]:

for col in null_cols:
    print(f"{col}: {data[col].nunique()} {data[col].unique()}")


Malware Indicators: 1 ['IoC Detected' nan]
Alerts/Warnings: 1 [nan 'Alert Triggered']
Proxy Information: 20148 ['150.9.97.135' nan '114.133.48.179' ... '60.51.30.46' '137.76.130.8'
 '112.169.115.139']
Firewall Logs: 1 ['Log Data' nan]
IDS/IPS Alerts: 1 [nan 'Alert Data']


In [6]:
data.groupby("Proxy Information")["Proxy Information"].count().sort_values(ascending=False)

Proxy Information
39.123.165.122    2
32.128.50.214     1
32.127.63.16      1
32.127.49.63      1
32.127.160.76     1
                 ..
166.56.182.249    1
166.49.59.72      1
166.48.160.244    1
166.44.193.97     1
166.72.179.119    1
Name: Proxy Information, Length: 20148, dtype: int64

can remove the colum proxy info because they're all unique values, could potentially split all the ip address segments into their own groups but I do not have enough domain knowledge to know if that would be useful

In [7]:
data = data.drop(null_cols, axis=1)
y_vals = data[target_value]
data = data.drop(target_value, axis=1)
data.head()

,Timestamp,Source IP Address,Destination IP Address,Source Port,Destination Port,Protocol,Packet Length,Packet Type,Traffic Type,Payload Data,Attack Type,Attack Signature,Action Taken,Severity Level,User Information,Device Information,Network Segment,Geo-location Data,Log Source
0,2023-05-30 06:33:58,103.216.15.12,84.9.164.252,31225,17616,ICMP,503,Data,HTTP,Qui natus odio asperiores nam. Optio nobis ius...,Malware,Known Pattern B,Logged,Low,Reyansh Dugal,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment A,"Jamshedpur, Sikkim",Server
1,2020-08-26 07:08:30,78.199.217.198,66.191.137.154,17245,48166,ICMP,1174,Data,HTTP,Aperiam quos modi officiis veritatis rem. Omni...,Malware,Known Pattern A,Blocked,Low,Sumer Rana,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment B,"Bilaspur, Nagaland",Firewall
2,2022-11-13 08:23:25,63.79.210.48,198.219.82.17,16811,53600,UDP,306,Control,HTTP,Perferendis sapiente vitae soluta. Hic delectu...,DDoS,Known Pattern B,Ignored,Low,Himmat Karpe,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT ...,Segment C,"Bokaro, Rajasthan",Firewall
3,2023-07-02 10:38:46,163.42.196.10,101.228.192.255,20018,32534,UDP,385,Data,HTTP,Totam maxime beatae expedita explicabo porro l...,Malware,Known Pattern B,Blocked,Medium,Fateh Kibe,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_11_5; ...,Segment B,"Jaunpur, Rajasthan",Firewall
4,2023-07-16 13:11:07,71.166.185.76,189.243.174.238,6131,26646,TCP,1462,Data,DNS,Odit nesciunt dolorem nisi iste iusto. Animi v...,DDoS,Known Pattern B,Blocked,Low,Dhanush Chad,Mozilla/5.0 (compatible; MSIE 5.0; Windows NT ...,Segment C,"Anantapur, Tripura",Firewall


## Spliting, imbalance, and scaling data

### Split data with 80/20 split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data.drop(target_value, axis=1), 
    y_vals, 
    test_size=0.2, 
    random_state=42)

### Scale data

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Applying PCA

In [ ]:
#apply PCA to reduce to 2 components
pca=PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
#visualize components of PCA
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', marker='o')
plt.title('PCA of Cyberattack Dataset')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(label='Cyberattack (1) / Normal (0)')
plt.show()